<a href="https://colab.research.google.com/github/SridharS-Square/Agentic_AI_Workshop/blob/main/Building%20Advanced%20Al%20Agents%20with%20AutoGen/Smart_Content_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyautogen google-generativeai

In [ ]:
# content_workflow.py

import os
import google.generativeai as genai
from autogen import AssistantAgent, UserProxyAgent

# --- 1. Setup and Configuration ---

def initialize_gemini():
    """
    Configures the Google Generative AI client using an API key.
    Best practice is to set the GOOGLE_API_KEY in your environment variables.
    """
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        print("⚠️ GOOGLE_API_KEY environment variable not found.")
        api_key = input("Please enter your Google API Key: ").strip()

    genai.configure(api_key=api_key)
    return api_key

API_KEY = initialize_gemini()

# Define a single, reusable LLM configuration for our agents
GEMINI_CONFIG = {
    "config_list": [
        {
            "model": "gemini-1.5-flash",
            "api_key": API_KEY,
            "api_type": "google",
        }
    ],
    "temperature": 0.7,  # Adding temperature for slightly more creative responses
}


# --- 2. Agent Definitions ---

# The agent responsible for writing the initial draft
writer_agent = AssistantAgent(
    name="Technical_Writer",
    llm_config=GEMINI_CONFIG,
    system_message="""You are a skilled Technical Writer specializing in AI. Your task is to produce a well-researched, engaging, and technically sound article based on the provided topic.
    Structure your content logically with clear headings. Ensure your explanations are easy for a tech-savvy audience to understand."""
)

# The agent responsible for reviewing and providing feedback
editor_agent = UserProxyAgent(
    name="Senior_Editor",
    human_input_mode="NEVER",  # The editor operates autonomously
    code_execution_config=False,
    llm_config=GEMINI_CONFIG, # This agent also uses the LLM to generate critiques
    system_message="""You are a Senior Editor. Your role is to critically evaluate the article draft submitted by the Technical Writer.
    Your feedback should be constructive, specific, and aimed at improving the article's quality.
    Check for:
    - Clarity and flow
    - Technical accuracy
    - Completeness based on the initial request
    - Engagement and tone
    Provide actionable suggestions for improvement. If the article is satisfactory, you can approve it by saying 'The article meets all criteria. Great work.'."""
)


# --- 3. Main Execution ---

def run_content_creation_workflow():
    """
    Initiates and manages the two-step content creation process.
    """
    print("\n" + "="*50)
    print("📝 Starting Agentic Content Creation Workflow 📝")
    print("="*50)

    # The initial task assigned to the writer
    initial_prompt = """
    Please draft an insightful blog post about the rise of Agentic AI.

    The article should cover the following key areas:
    1.  **What is Agentic AI?**: A clear, concise definition.
    2.  **Core Components**: Explain how they work (e.g., planning, tools, memory).
    3.  **Current Challenges**: Discuss limitations like reliability and predictability.
    4.  **Future Opportunities**: Explore the potential impact across various industries.

    Please begin the draft.
    """

    # The Editor initiates the conversation by giving the Writer the task
    editor_agent.initiate_chat(
        recipient=writer_agent,
        message=initial_prompt,
        max_turns=4 # Limiting to a few rounds of feedback
    )

    print("\n" + "="*50)
    print("✅ Workflow Complete.")
    print("="*50)


if __name__ == "__main__":
    run_content_creation_workflow()